In [ ]:
# R1_Simulate_Roman_Straylight
# Alejandro S. Borlaff - NASA Ames Research Center. 
# STA N245-312 - a.s.borlaff@nasa.gov

# Lets generate an example Roman/WFI image with realistic stray-light from the stars all-sky. 
"""

import os
import glob
from tqdm import tqdm
import straycor as sc
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

from reproject import reproject_interp
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord  # High-level coordinates
from astropy.coordinates import ICRS
import astropy.wcs as astropy_wcs

import coord
import galsim
import galsim.roman as roman

from astropy.time import Time
from astropy.io import fits
from astropy.coordinates import SkyCoord  # High-level coordinates
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
from astropy.coordinates import Angle, Latitude, Longitude  # Angles
import astropy.units as u
from astropy import constants as const
from datetime import datetime


"""

import os
import glob
import numpy as np
import rosalia as rs
from astropy.io import fits
from astropy.io import fits
import astropy.wcs as astropy_wcs
from tqdm import tqdm
import matplotlib.pyplot as plt

rosalia_git_directory = "/Users/aborlaff/NASA/ROSALIA/" # Here put the base directory of the local copy of the ROSALIA repository. 
# Download it from here: 
# https://github.com/Borlaff/STRAYCOR
plt.style.use('dark_background')


In [ ]:
os.system("romanisim-make-image --radec 200 -23 RST_WFI_ROSALIA_ra200_dec-23_test_SCA{}.asdf --roll 0 --sca -1 --bandpass F158 --level 2 --usecrds")



In [ ]:
import asdf
input_name = "/Users/aborlaff/NASA/ROSALIA/notebooks/DEVEL/RST_WFI_ROSALIA_ra200_dec-23_test_SCAwfi01.asdf"
input_asdf = asdf.open(input_name)
input_asdf["roman"]["meta"]

### Test 1: Static stars

In [ ]:
# EXAMPLE OF THE USE OF EXPOSURE INSPECTOR # 
# exposure_identity = sc.utils.exposure_inspector("/Users/aborlaff/NASA/ROSALIA_DEPOT/ASDF_SIMULATION/RST_WFI_test_SCAWFI01.asdf")
# print(exposure_identity)

catalog = {"ra": np.array([200.135, 200.137]), "dec": np.array([-22.896, -22.896]), "mag": np.array([-1, -1.2])}

# Test 1: Static stars. One bright far away, one dimmer close. 
if True:
    exposure_name_list = glob.glob("/Users/aborlaff/NASA/ROSALIA/notebooks/DEVEL/RST_WFI_ROSALIA_ra200_dec-23_test_SCAwfi*.asdf")
    output_name = rs.roman._test_run_rosalia_asdf(catalog, exposure_name_list, verbose=False, clean=False)



In [ ]:
exposure_identify = rs.utils.exposure_inspector("/Users/aborlaff/NASA/ROSALIA_DEPOT/ASDF_SIMULATION/RST_WFI_test_SCAWFI01.asdf")
exposure_identify

### Test 2: Star moving across the detector 

In [ ]:
# Let's make a movie about a star moving over the lightshield of Roman/WFI. 
# To do this, we first generate a series of coordinates drawing a rectangle over the focal plane array of an existing
# simulated Roman/WFI exposure. 

ra_list = np.concatenate([np.linspace(199.85,200.144,20),
                          np.linspace(200.144,200.144,20), 
                          np.linspace(200.144,199.85,20), 
                          np.linspace(199.85,199.85,20)]) 

dec_list = np.concatenate([np.linspace(-22.896,-22.896,20),
                           np.linspace(-22.896,-23.04,20), 
                           np.linspace(-23.04,-23.04,20), 
                           np.linspace(-23.04,-22.896,20)])

synthetic_mag_outside= -1

# For each position of the star, let's make a mosaic of the stray-light
# exposure_name_list = glob.glob("/Users/aborlaff/NASA/ROSALIA_DEPOT/ASDF_SIMULATION/RST_WFI_test_SCAWFI*.asdf")
exposure_name_list = glob.glob("/Users/aborlaff/NASA/ROSALIA/notebooks/DEVEL/RST_WFI_ROSALIA_ra200_dec-23_test_SCAwfi*.asdf")


if True:
    for i in tqdm(range(len(ra_list))):
        catalog = {"ra": np.array([ra_list[i]]), "dec": np.array([dec_list[i]]), "mag": np.array([-1])}
        rs.roman._test_run_rosalia_asdf(catalog, exposure_name_list, verbose=False)

        # At the end of each simulation, move the mosaic to a new name so we do not overwrite them all. 
        rs.utils.execute_cmd("mv coadd_scaled.fits moving_mag_1_star_v6" + str(i).zfill(3) + ".fits")


# When all the simulations are done, we can plot them to show the gradients of light across the FPA
# and also the position of the star over the NDI map of the lightshield.

if True:

    for i in tqdm(range(len(ra_list))):
        # Axis 1 
        stray_name = "moving_mag_1_star_v6" + str(i).zfill(3) + ".fits"
        stray_fits = fits.open(stray_name)
        fig, axs = plt.subplots(1, 2, figsize=(14, 10)) 
    
        w = astropy_wcs.WCS(header=stray_fits[1].header, fobj=stray_fits, naxis=2)

        stray_fits[1].data[stray_fits[1].data == 0] = np.nan
        im = axs[0].imshow(stray_fits[1].data, vmin=150, vmax=800, origin='lower', cmap="RdYlBu_r")
        plt.colorbar(im, location='top', label='Stray-light (s$^{-1}$)')

        #ax.plot(ra_list, dec_list, transform=ax.get_transform('world'))
        axs[0].set_xlabel("Y")
        axs[0].set_ylabel("X")

        # Axis 2 
        axs[1] = plt.subplot(122)
        ndi_name = "/Users/aborlaff/NASA/STRAYCOR/notebooks/ROSALIA/NDI_m05deg_mean.fits"
        ndi_name_header = fits.open("/Users/aborlaff/NASA/ROSALIA_DEPOT/NDI_HEAVY_FILES/CUBES/1-deg_SCA_10_SUB_X-17.5_Y-12.5.fits")[0].header

        ndi_fits = fits.open(ndi_name)
        #ndi_map = np.mean(ndi_fits[0].data, axis=0)
        im = axs[1].imshow(ndi_fits[0].data, vmin=0.0, vmax=1, origin='lower', cmap="RdYlBu_r")

        ndi_name_header["CRVAL1"] = 200
        ndi_name_header["CRVAL2"] = -23
        w = astropy_wcs.WCS(header=ndi_name_header, fobj=ndi_fits, naxis=2)
        x_stars, y_stars = w.wcs_world2pix(ra_list, dec_list, 0)
        axs[1].scatter(x_stars[i], y_stars[i], marker="*", color="red", alpha=0.9, s=500)

        #x_stars, y_stars = w.wcs_world2pix([196], [-17.5], 0)
        #axs[1].scatter(x_stars[i], y_stars[i], marker="o", color="black")

        axs[1].set_title("Stray-light source location on Focal Plane\n")
 
        plt.tight_layout()
        plt.savefig(stray_name.replace(".fits", ".png"), dpi=100)
        plt.show()

        #os.system("magick -delay 10 -loop 0 moving_star*.png moving_star_v1.gif")
    os.system("magick -delay 10 -loop 0 moving_mag_1_star_v6*.png moving_star_v6.gif")

### For each SCA in Roman/WFI, we find the stars around and estimate the stray-light per pixel.
#### WARNING: Since we need to query the Gaia database, this takes a lot of time.
#### It depends 50% on your connection, and another 50% is the Gaia Archive response time.

In [ ]:
roman_dummy_image_list = glob.glob("/Users/aborlaff/NASA/ROSALIA_DEPOT/ASDF_SIMULATION/RST_WFI_test_SCAWFI*.asdf")
print(roman_dummy_image_list)
import pandas as pd

input_catalog = pd.DataFrame({"ra": np.array([199.85]), "dec": np.array([-22.896]), "mag_lambda": 0})

roman_dummy_image = roman_dummy_image_list[0]
if True:
    straylight_dict  = sc.correct.main_offender(input_name=roman_dummy_image,
                                                radius=0.5, g_mag_max=15, step=400, 
                                                verbose=True)

### Now we combine the straylight Roman/WFI image to the simulated one

In [ ]:
print(sc.detectors.mu2fe(30, instrument="WFI", filter_name="F146", telescope="Roman"))
print(sc.detectors.mu2fe(30, instrument="ACS", filter_name="F475W", telescope="Hubble"))
print(sc.detectors.mu2fe(30, instrument="ACS", filter_name="F850LP", telescope="Hubble"))
filter_db = sc.telescopes.Hubble.get_filter(instrument="ACS", filter_name="F475W")
plt.plot(filter_db["wavelength_bins"], filter_db["transmission_bins"])

In [ ]:
filter_db = sc.telescopes.Roman.get_filter(instrument="WFI", filter_name="F146")
plt.plot(filter_db["wavelength_bins"], filter_db["transmission_bins"])

In [ ]:
print(straylight_dict["straylevel_list"][0].shape)
mu = sc.detectors.fe2mu(straylight_dict["straylevel_list"][0]/500/500, instrument="WFI", filter_name="F146", telescope="Roman")
plt.imshow(mu)
plt.colorbar()

In [ ]:
# Open a Roman romanisim simulated dummy image


In [ ]:
# Open the Roman dummy image 
roman_dummy_stray = fits.open(straylight_dict["output_name"])
roman_dummy = fits.open(roman_dummy_image)

for i in tqdm(range(len(roman_dummy)-1)):    
    roman_dummy[i+1].data = roman_dummy[i+1].data + roman_dummy_stray[i+1].data

roman_dummy.verify("silentfix")
roman_dummy.writeto(roman_dummy_image , overwrite=True)



In [ ]:
# We drizzle the multiextension image to a single array.
# In the future, we will be able to use drizzlepac in a similar way to this:
# from drizzlepac import astrodrizzle
# astrodrizzle.AstroDrizzle(input="tutorial_roman_dummy_with_galaxy.fits")
# But Roman is not supported yet.
# For this test, we will do a simpler version using Reproject. 
# https://reproject.readthedocs.io/en/stable/api/reproject.mosaicking.reproject_and_coadd.html
# https://reproject.readthedocs.io/en/stable/mosaicking.html
from reproject import mosaicking
from reproject import reproject_interp
from reproject.mosaicking import find_optimal_celestial_wcs
from drizzle import drizzle # pip install git+https://github.com/spacetelescope/drizzle


#roman_dummy_fits = fits.open(roman_dummy_image.replace(".fits", "_with_galaxy.fits"))
wcs_out, shape_out = find_optimal_celestial_wcs(roman_dummy[1:])
reference_header = wcs_out.to_header()
reference_header["NAXIS"] = 2
reference_header["NAXIS1"] = shape_out[1]
reference_header["NAXIS2"] = shape_out[0]
reference_header

# mosaicking.reproject_and_coadd(roman_dummy_fits[1:], output_projection=wcs_out, shape_out=shape_out, reproject_function=reproject_interp)
#dummy_drizzle_image = np.zeros(shape_out)
#

if True:
    # Get the WCS for the output image
    #hdulist = fits.open(reference)
    #reference_wcs = wcs.WCS(hdulist[1].header)

    # Initialize the output with the WCS
    driz = drizzle.Drizzle(outwcs=astropy_wcs.WCS(reference_header))

    # Combine the input images into on drizzle image
    for SCA_i in tqdm(range(len(roman_dummy)-1)):
        driz.add_fits_file(roman_dummy_image+"["+ str(SCA_i+1) + "]")

    # Write the drizzled image out
    driz.write("test_drizzle.fits")

In [ ]:
#os.system("swarp -c output/swarp.conf output/demo13*SCA*with_stray.fits") 
#os.system("echo -e '\a'")
#os.system("mv coadd.fits galaxy_with_single_star.fits")




#roman_dummy_fits = fits.open(roman_dummy_image.replace(".fits", "_with_galaxy.fits"))
wcs_out, shape_out = find_optimal_celestial_wcs(roman_dummy_stray[1:])
reference_header = wcs_out.to_header()
reference_header["NAXIS"] = 2
reference_header["NAXIS1"] = shape_out[1]
reference_header["NAXIS2"] = shape_out[0]
reference_header

# mosaicking.reproject_and_coadd(roman_dummy_fits[1:], output_projection=wcs_out, shape_out=shape_out, reproject_function=reproject_interp)
#dummy_drizzle_image = np.zeros(shape_out)
#

if True:
    # Get the WCS for the output image
    #hdulist = fits.open(reference)
    #reference_wcs = wcs.WCS(hdulist[1].header)

    # Initialize the output with the WCS
    driz = drizzle.Drizzle(outwcs=astropy_wcs.WCS(reference_header))

    # Combine the input images into on drizzle image
    for SCA_i in tqdm(range(len(roman_dummy_stray)-1)):
        driz.add_fits_file(straylight_dict["output_name"]+"["+ str(SCA_i+1) + "]")

    # Write the drizzled image out
    driz.write("test_drizzle_stray.fits")